# RL-token-minimize — full pipeline on Kaggle

Trains a Qwen3-0.6B tool-use code-repair agent (LoRA SFT → GRPO) and measures
whether RL buys **fewer tokens at the same repair quality**, with and without
reasoning ("thinking") enabled.

**Setup:** Settings → Accelerator → **GPU T4 x2** (only GPU 0 is used; HF Trainer's
DataParallel breaks the PEFT setup) and Settings → Internet → **On**.

## This notebook is resumable — expect to run it over several sessions

A Kaggle session is capped at 12 h and the full pipeline is ~25-30 h of GPU time.
Every expensive stage is **skipped when its artifact already exists**, so:

1. Run the notebook. It completes as many stages as fit in the session.
2. Save Version → let it finish.
3. Next session: **Add Data → Your Work → this notebook's output**, then run again.
   The restore cell copies `checkpoints/` out of `/kaggle/input/` and the run
   continues exactly where it stopped.
4. Repeat until the last cell prints `PIPELINE COMPLETE`.

Nothing is recomputed twice, and the held-out split is deterministic (seed 0), so
numbers stay comparable across sessions.

## Measured cost (T4, from the 2026-07-17 run)

| Stage | Cost |
|---|---|
| setup + tests + data prep | ~2 min |
| LoRA SFT (121 traces, 3 epochs) | ~13 min |
| eval n=40, no thinking | 4–9 min |
| eval n=40, thinking | ~20–40 min (longer generations) |
| RL, no thinking | **2.24 min/step** → 200 steps = 7.5 h |
| RL, thinking | measured by the smoke cell in §4 |

Qwen3 keeps reasoning in context across tool calls within one task, so thinking
episodes have longer prompts and cost noticeably more per step. The smoke cell
measures the real rate before committing to a full run.

## 0 · Setup, restore, configuration

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
%env PYTORCH_ALLOC_CONF=expandable_segments:True
%cd /kaggle/working
!rm -rf RL-token-minimize
!git clone --depth 1 https://github.com/augustoFranke/RL-token-minimize.git
%cd RL-token-minimize
!mkdir -p checkpoints/figs
!pip uninstall -q -y torchao
!pip install -q -U "transformers>=5.13" "trl>=1.7.1" "peft>=0.19.1" "datasets>=5.0.0" "accelerate>=1.14.0" pytest
!pip install -q -e . --no-deps

import subprocess, torch
print("commit ", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())
print("torch  ", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu    ", props.name, f"| {props.total_memory / 2**30:.1f} GiB")
    print("bf16   ", torch.cuda.is_bf16_supported(including_emulation=False),
          "native (False => the run uses fp32)")

In [ ]:
# Restore checkpoints from a previous session's output, if attached as input.
import glob, shutil
from pathlib import Path

restored = 0
for src_root in sorted(glob.glob("/kaggle/input/*/checkpoints")) + \
                sorted(glob.glob("/kaggle/input/*/*/checkpoints")):
    n_here = 0
    for src in Path(src_root).rglob("*"):
        if not src.is_file():
            continue
        dst = Path("checkpoints") / src.relative_to(src_root)
        if dst.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        n_here += 1
    print(f"restored {n_here} file(s) from {src_root}")
    restored += n_here

print(f"\n{restored} file(s) restored; present artifacts:")
for p in sorted(Path("checkpoints").glob("*")):
    print("   ", p.name + ("/" if p.is_dir() else ""))
if not restored:
    print("    (first session, or no prior output attached)")

In [ ]:
%%writefile stage.sh
#!/bin/bash
# stage.sh <name> <artifact> <command...>
# Runs <command> unless <artifact> already exists; records wall time.
name=$1; artifact=$2; shift 2
if [ -e "$artifact" ]; then
  echo "== skip $name — $artifact exists"
  exit 0
fi
echo "== run $name -> $artifact"
echo "   \$ $*"
start=$(date +%s)
"$@"
rc=$?
elapsed=$(( $(date +%s) - start ))
mkdir -p checkpoints
echo "{\"stage\": \"$name\", \"seconds\": $elapsed, \"rc\": $rc}" >> checkpoints/timings.jsonl
if [ $rc -eq 0 ]; then
  echo "== done $name in $((elapsed / 60))m $((elapsed % 60))s"
else
  echo "== FAILED $name (rc=$rc) after $((elapsed / 60))m $((elapsed % 60))s"
fi
exit $rc

In [ ]:
# Experiment configuration. Runs 2 and 3 differ from run 1 ONLY in thinking and
# how tokens are charged, so the comparison stays clean.
RL_STEPS_NOTHINK = 200      # run 1 (control); 2.24 min/step measured => ~7.5 h
RL_STEPS_THINK   = 100      # runs 2 and 3; size with the smoke cell in §4
MAX_NEW_TOKENS   = 1024     # identical across runs
MAX_TURNS        = 10       # identical across runs

import os
os.environ.update(RL_STEPS_NOTHINK=str(RL_STEPS_NOTHINK), RL_STEPS_THINK=str(RL_STEPS_THINK),
                  MAX_NEW_TOKENS=str(MAX_NEW_TOKENS), MAX_TURNS=str(MAX_TURNS))
print(f"run1  no thinking,      cost=all   : {RL_STEPS_NOTHINK} steps")
print(f"run2  thinking,         cost=all   : {RL_STEPS_THINK} steps")
print(f"run3  thinking,         cost=final : {RL_STEPS_THINK} steps")
print(f"max_new_tokens={MAX_NEW_TOKENS}  max_turns={MAX_TURNS}")

In [ ]:
# Analysis helpers, used by the gate (§3) and the analysis (§6).
import json, math
from pathlib import Path

def rows_of(path):
    """Per-task rows from an evaluate.py JSONL (without the trailing summary)."""
    if not Path(path).exists():
        return []
    recs = [json.loads(line) for line in open(path) if line.strip()]
    return [r for r in recs if "summary" not in r]

def summary_of(path):
    if not Path(path).exists():
        return None
    for line in open(path):
        if line.strip():
            rec = json.loads(line)
            if "summary" in rec:
                return rec["summary"]
    return None

def mean(xs, default=float("nan")):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else default

def wilson_ci(k, n, z=1.96):
    """95% CI for a proportion — n=40 is small, so always report the interval."""
    if n == 0:
        return (float("nan"), float("nan"))
    p, d = k / n, 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

def sign_test(a, b):
    """Two-sided exact binomial on a vs b discordant counts (also McNemar)."""
    n = a + b
    if n == 0:
        return 1.0
    k = min(a, b)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2**n)

def rolling(xs, w=15):
    return [mean(xs[max(0, i - w + 1): i + 1]) for i in range(len(xs))]

print("helpers ready")

In [ ]:
!python -m pytest -q

In [ ]:
!bash stage.sh data data/tasks_eval.jsonl python scripts/prepare_data.py
!wc -l data/*.jsonl

## 1 · SFT

Teaches the tool-call format from traces derived mechanically from the reference
patches. The goal is a policy RL can improve on, not a strong model.

In [ ]:
!bash stage.sh sft checkpoints/sft/adapter_model.safetensors python scripts/train_sft.py

## 2 · Baselines on the 40 held-out tasks

Four greedy, reproducible reference points: base and post-SFT, each with and
without thinking. `sft` is the bar run 1 must clear; `sft+think` is the bar for
runs 2 and 3.

In [ ]:
!bash stage.sh eval_base checkpoints/eval_base.jsonl \
    python scripts/evaluate.py --no-thinking --out checkpoints/eval_base.jsonl

!bash stage.sh eval_base_think checkpoints/eval_base_think.jsonl \
    python scripts/evaluate.py --thinking --out checkpoints/eval_base_think.jsonl

In [ ]:
!bash stage.sh eval_sft checkpoints/eval_sft.jsonl \
    python scripts/evaluate.py --adapter checkpoints/sft --no-thinking \
    --out checkpoints/eval_sft.jsonl

!bash stage.sh eval_sft_think checkpoints/eval_sft_think.jsonl \
    python scripts/evaluate.py --adapter checkpoints/sft --thinking \
    --out checkpoints/eval_sft_think.jsonl

## 3 · Thinking gate — check before spending ~8 h per run

SFT trained the adapter in **non-thinking** format (that template puts an empty
`<think></think>` into the prompt). Turning thinking on at RL time asks for a
format the adapter has not seen since fine-tuning, so verify first that the model
actually reasons *and* still reaches a tool call.

A local probe on the **base** model produced 253 reasoning tokens and then ran
past a 512-token cap without emitting a tool call — so both failure modes below
are real.

- **PASS** — reasoning appears and protocol errors stay low: runs 2/3 are meaningful.
- **FAIL, no reasoning** — the adapter skips `<think>`; runs 2/3 would duplicate
  run 1. Fix by prefilling `<think>` into the generation prompt.
- **FAIL, protocol errors ≈ 1** — reasoning overruns `MAX_NEW_TOKENS` before any
  tool call. Fix by raising `MAX_NEW_TOKENS`.

In [ ]:
think = rows_of("checkpoints/eval_sft_think.jsonl")
plain = rows_of("checkpoints/eval_sft.jsonl")

if not think:
    print("run the §2 eval cells first")
else:
    n = len(think)
    with_reasoning = sum(r["thinking_tokens"] > 0 for r in think)
    proto = mean(r["protocol_error"] for r in think)
    truncated = sum(r["tokens"] >= MAX_NEW_TOKENS for r in think)

    print(f"tasks that reasoned  : {with_reasoning}/{n}")
    print(f"mean reasoning tokens: {mean(r['thinking_tokens'] for r in think):.1f}")
    print(f"mean total tokens    : {mean(r['tokens'] for r in think):.1f}")
    print(f"hit the token cap    : {truncated}/{n}")
    print(f"protocol error rate  : {proto:.3f}")
    print(f"pass rate            : {mean(r['passed'] for r in think):.3f}")
    if plain:
        print(f"\nno-thinking reference: pass {mean(r['passed'] for r in plain):.3f}, "
              f"tokens {mean(r['tokens'] for r in plain):.1f}, "
              f"protocol {mean(r['protocol_error'] for r in plain):.3f}")

    print()
    if with_reasoning == 0:
        print("GATE FAIL — no reasoning emitted; runs 2/3 would duplicate run 1.")
    elif proto > 0.5:
        print(f"GATE FAIL — {proto:.0%} protocol errors; raise MAX_NEW_TOKENS and re-run.")
    else:
        print("GATE PASS — continue to the thinking RL runs.")

## 4 · RL

Three GRPO runs, identical except for reasoning and how tokens are charged:

| run | thinking | token cost | question it answers |
|---|---|---|---|
| `rl_run1` | off | every generated token | does RL cut tokens at equal quality? |
| `rl_run2` | on | every generated token | does charging for reasoning suppress it? |
| `rl_run3` | on | post-reasoning tokens only | does free reasoning inflate it? |

Run the smoke cell first — it measures the real per-step cost of thinking so you
can size `RL_STEPS_THINK` against the 12 h session cap.

In [ ]:
%%time
# Cost probe: 10 thinking steps, then extrapolate.
!bash stage.sh rl_think_smoke checkpoints/rl_think_smoke/final \
    python scripts/train_rl.py --adapter checkpoints/sft --thinking --token-cost all \
    --steps 10 --max-new-tokens $MAX_NEW_TOKENS --max-turns $MAX_TURNS \
    --output checkpoints/rl_think_smoke

In [ ]:
timings = [json.loads(l) for l in open("checkpoints/timings.jsonl")
           if l.strip()] if Path("checkpoints/timings.jsonl").exists() else []
smoke = next((t for t in timings if t["stage"] == "rl_think_smoke" and t["rc"] == 0), None)

if smoke:
    per_step = smoke["seconds"] / 10
    print(f"thinking RL : {per_step / 60:.2f} min/step")
    print(f"no-thinking : 2.24 min/step (measured 2026-07-17)")
    print(f"\nprojected wall time:")
    for steps in (50, 100, 150, 200):
        h = steps * per_step / 3600
        print(f"  {steps:3d} steps -> {h:5.2f} h per run   ({2 * h:5.2f} h for runs 2+3)")
    print(f"\nRL_STEPS_THINK={RL_STEPS_THINK} costs ~{RL_STEPS_THINK * per_step / 3600:.1f} h/run.")
    print("If that exceeds what is left of the 12 h session, lower RL_STEPS_THINK")
    print("in §0 and re-run — or let this session end and finish in the next one.")
else:
    smoke_log = Path("checkpoints/rl_think_smoke/log.jsonl")
    print("no smoke timing recorded (stage was skipped).",
          "Existing log:" if smoke_log.exists() else "")
    if smoke_log.exists():
        recs = [json.loads(l) for l in open(smoke_log) if l.strip()]
        print(f"  {len(recs)} steps, last: {recs[-1]}")

In [ ]:
# Resume support. train_rl.py checkpoints every 20 steps and appends to log.jsonl,
# so a run killed by the 12 h session cap continues from its last checkpoint
# instead of throwing away hours of GPU time.
import shutil

def rl_resume(run_dir, total_steps):
    """-> (adapter to start from, steps still to run, steps already done)."""
    d = Path(run_dir)
    saved = sorted((int(p.name.split("_")[1]), p) for p in d.glob("step_*")) if d.exists() else []
    if not saved:
        return "checkpoints/sft", total_steps, 0
    done, path = saved[-1]
    remaining = max(0, total_steps - done)
    if remaining == 0 and not (d / "final").exists():
        shutil.copytree(path, d / "final")   # finished but killed before the final save
        print(f"{run_dir}: {done} steps already done; promoted {path.name} to final")
    return str(path), remaining, done

def plan(run_dir, total_steps):
    adapter, remaining, done = rl_resume(run_dir, total_steps)
    if Path(run_dir, "final").exists():
        print(f"{run_dir}: complete — will be skipped")
    elif done:
        print(f"{run_dir}: resuming from {adapter} ({done} done, {remaining} to go)")
    else:
        print(f"{run_dir}: fresh start from {adapter} ({remaining} steps)")
    return adapter, str(remaining)

# Bound as plain Python names: the `!` cells below interpolate $NAME from the
# notebook namespace, so these must exist as variables, not just in os.environ.
R1_ADAPTER, R1_STEPS = plan("checkpoints/rl_run1", RL_STEPS_NOTHINK)
R2_ADAPTER, R2_STEPS = plan("checkpoints/rl_run2", RL_STEPS_THINK)
R3_ADAPTER, R3_STEPS = plan("checkpoints/rl_run3", RL_STEPS_THINK)

In [ ]:
%%time
# Run 1 — control: no thinking, every token charged.
!bash stage.sh rl_run1 checkpoints/rl_run1/final \
    python scripts/train_rl.py --adapter $R1_ADAPTER --no-thinking --token-cost all \
    --steps $R1_STEPS --max-new-tokens $MAX_NEW_TOKENS --max-turns $MAX_TURNS \
    --output checkpoints/rl_run1
!tail -3 checkpoints/rl_run1/log.jsonl

In [ ]:
%%time
# Run 2 — thinking, every generated token charged (raw token penalty).
!bash stage.sh rl_run2 checkpoints/rl_run2/final \
    python scripts/train_rl.py --adapter $R2_ADAPTER --thinking --token-cost all \
    --steps $R2_STEPS --max-new-tokens $MAX_NEW_TOKENS --max-turns $MAX_TURNS \
    --output checkpoints/rl_run2
!tail -3 checkpoints/rl_run2/log.jsonl

In [ ]:
%%time
# Run 3 — thinking, only post-reasoning tokens charged (reasoning is free).
!bash stage.sh rl_run3 checkpoints/rl_run3/final \
    python scripts/train_rl.py --adapter $R3_ADAPTER --thinking --token-cost final \
    --steps $R3_STEPS --max-new-tokens $MAX_NEW_TOKENS --max-turns $MAX_TURNS \
    --output checkpoints/rl_run3
!tail -3 checkpoints/rl_run3/log.jsonl

## 5 · Held-out evaluation of every RL checkpoint

Training logs report *sampled* rollouts on *training* tasks. These are greedy runs
on the 40 held-out tasks, and they are what the conclusions rest on. `rl_run1` is
also evaluated at step 100 so it can be compared with the thinking runs at equal
step count.

In [ ]:
!bash stage.sh eval_rl1 checkpoints/eval_rl1.jsonl \
    python scripts/evaluate.py --adapter checkpoints/rl_run1/final --no-thinking \
    --out checkpoints/eval_rl1.jsonl

# step_100 only exists if run 1 was not resumed past it; skip quietly if absent.
!test -d checkpoints/rl_run1/step_100 \
    && bash stage.sh eval_rl1_at100 checkpoints/eval_rl1_at100.jsonl \
       python scripts/evaluate.py --adapter checkpoints/rl_run1/step_100 --no-thinking \
       --out checkpoints/eval_rl1_at100.jsonl \
    || echo "== skip eval_rl1_at100 — checkpoints/rl_run1/step_100 not present" 

In [ ]:
!bash stage.sh eval_rl2 checkpoints/eval_rl2_think.jsonl \
    python scripts/evaluate.py --adapter checkpoints/rl_run2/final --thinking \
    --out checkpoints/eval_rl2_think.jsonl

!bash stage.sh eval_rl3 checkpoints/eval_rl3_think.jsonl \
    python scripts/evaluate.py --adapter checkpoints/rl_run3/final --thinking \
    --out checkpoints/eval_rl3_think.jsonl

## 6 · Analysis

Reads only saved artifacts, so it re-runs safely in a later session (or locally,
without a GPU). Missing stages are reported rather than crashing.

In [ ]:
CONFIGS = [
    ("base",       "checkpoints/eval_base.jsonl",       "off"),
    ("base+think", "checkpoints/eval_base_think.jsonl", "on"),
    ("sft",        "checkpoints/eval_sft.jsonl",        "off"),
    ("sft+think",  "checkpoints/eval_sft_think.jsonl",  "on"),
    ("rl1@100",    "checkpoints/eval_rl1_at100.jsonl",  "off"),
    ("rl1",        "checkpoints/eval_rl1.jsonl",        "off"),
    ("rl2+think",  "checkpoints/eval_rl2_think.jsonl",  "on"),
    ("rl3+think",  "checkpoints/eval_rl3_think.jsonl",  "on"),
]

loaded = {}
for name, path, think in CONFIGS:
    rows = rows_of(path)
    if rows:
        loaded[name] = {"rows": {r["task_id"]: r for r in rows},
                        "think": think, "summary": summary_of(path)}

cols = (f"{'config':<12}{'think':>6}{'n':>4}{'pass':>7}{'95% CI':>14}{'tok':>8}"
        f"{'reason':>8}{'final':>8}{'tok|pass':>10}{'tools':>7}{'proto':>7}")
print(cols)
print("-" * len(cols))
for name, _, _ in CONFIGS:
    if name not in loaded:
        print(f"{name:<12}{'(not run yet)':>30}")
        continue
    rows = list(loaded[name]["rows"].values())
    n = len(rows)
    k = sum(r["passed"] for r in rows)
    lo, hi = wilson_ci(k, n)
    reason = mean(r["thinking_tokens"] for r in rows)
    total = mean(r["tokens"] for r in rows)
    passing = [r["tokens"] for r in rows if r["passed"]]
    print(f"{name:<12}{loaded[name]['think']:>6}{n:>4}{k / n:>7.3f}"
          f"{f'[{lo:.2f},{hi:.2f}]':>14}{total:>8.1f}{reason:>8.1f}{total - reason:>8.1f}"
          f"{mean(passing):>10.1f}{mean(r['tool_calls'] for r in rows):>7.2f}"
          f"{mean(r['protocol_error'] for r in rows):>7.3f}")

print("\ntok = mean generated tokens/task | reason = tokens inside <think> |"
      " final = tok - reason")
print("tok|pass = mean tokens on solved tasks only | proto = protocol-error rate")

In [ ]:
# The headline claim: fewer tokens at the SAME quality.
# Tokens are compared only over tasks BOTH configs solved — the fair comparison.

def compare(a_name, b_name):
    if a_name not in loaded or b_name not in loaded:
        print(f"{b_name} vs {a_name}: artifact missing — skipped\n")
        return None
    A, B = loaded[a_name]["rows"], loaded[b_name]["rows"]
    shared = sorted(set(A) & set(B))
    both = [t for t in shared if A[t]["passed"] and B[t]["passed"]]
    gained = [t for t in shared if B[t]["passed"] and not A[t]["passed"]]
    lost = [t for t in shared if A[t]["passed"] and not B[t]["passed"]]

    print(f"=== {b_name} vs {a_name}   (n={len(shared)} shared tasks) ===")
    print(f"quality : {sum(A[t]['passed'] for t in shared)} -> "
          f"{sum(B[t]['passed'] for t in shared)} passed "
          f"(+{len(gained)} / -{len(lost)}, McNemar p={sign_test(len(lost), len(gained)):.3f})")
    if gained:
        print(f"  gained: {', '.join(gained)}")
    if lost:
        print(f"  lost  : {', '.join(lost)}")

    result = {"pair": f"{b_name}_vs_{a_name}", "n_shared": len(shared),
              "gained": gained, "lost": lost, "n_both_passed": len(both)}
    if not both:
        print("  no commonly-solved task — token comparison impossible\n")
        return result

    deltas = sorted(B[t]["tokens"] - A[t]["tokens"] for t in both)
    a_tok = mean(A[t]["tokens"] for t in both)
    b_tok = mean(B[t]["tokens"] for t in both)
    shorter = sum(d < 0 for d in deltas)
    median = deltas[len(deltas) // 2]
    pct = (b_tok - a_tok) / a_tok * 100 if a_tok else float("nan")
    print(f"tokens on the {len(both)} task(s) both solved:")
    print(f"  {a_name} {a_tok:.1f} -> {b_name} {b_tok:.1f}   ({b_tok - a_tok:+.1f}, {pct:+.1f}%)")
    print(f"  median delta {median:+.0f} | shorter on {shorter}/{len(both)} "
          f"(sign test p={sign_test(shorter, len(both) - shorter):.3f})")
    print()
    result.update({"tokens_a": a_tok, "tokens_b": b_tok, "pct_change": pct,
                   "median_delta": median, "shorter_fraction": shorter / len(both)})
    return result

comparisons = [c for c in (
    compare("base", "sft"),            # did SFT work?
    compare("sft", "rl1"),             # THE headline: RL vs SFT, no thinking
    compare("sft+think", "rl2+think"),  # RL with reasoning charged
    compare("sft+think", "rl3+think"),  # RL with reasoning free
    compare("rl2+think", "rl3+think"),  # does charging reasoning matter?
    compare("rl1", "rl2+think"),       # is reasoning worth it at all?
) if c]

In [ ]:
# Training curves for every RL run that produced a log.
import matplotlib.pyplot as plt

RUNS = [("rl_run1 (no thinking)",          "checkpoints/rl_run1/log.jsonl"),
        ("rl_run2 (thinking, cost=all)",   "checkpoints/rl_run2/log.jsonl"),
        ("rl_run3 (thinking, cost=final)", "checkpoints/rl_run3/log.jsonl")]

logs = {}
for name, path in RUNS:
    if Path(path).exists():
        recs = [json.loads(l) for l in open(path) if l.strip()]
        if recs:
            logs[name] = recs

if logs:
    metrics = [("mean_reward", "mean reward"), ("pass_rate", "pass rate (sampled, train)"),
               ("mean_tokens", "mean tokens"), ("mean_thinking_tokens", "reasoning tokens")]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for ax, (key, label) in zip(axes.flat, metrics):
        for name, recs in logs.items():
            # position, not r["step"]: a resumed run restarts its step counter
            steps = list(range(len(recs)))
            vals = [r[key] for r in recs]
            ax.plot(steps, vals, alpha=0.18, lw=1)
            ax.plot(steps, rolling(vals), lw=2, label=name)
        ax.set_title(label)
        ax.set_xlabel("step")
        ax.grid(alpha=0.3)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle("GRPO training — faint: per step, bold: 15-step rolling mean")
    fig.tight_layout()
    fig.savefig("checkpoints/figs/training_curves.png", dpi=120)
    plt.show()

    print(f"{'run':<32}{'steps':>7}{'pass first->last 25%':>24}{'tokens first->last 25%':>26}")
    for name, recs in logs.items():
        q = max(1, len(recs) // 4)
        first, last = recs[:q], recs[-q:]
        p0, p1 = mean(r["pass_rate"] for r in first), mean(r["pass_rate"] for r in last)
        t0, t1 = mean(r["mean_tokens"] for r in first), mean(r["mean_tokens"] for r in last)
        print(f"{name:<32}{len(recs):>7}{f'{p0:.3f} -> {p1:.3f}':>24}"
              f"{f'{t0:.0f} -> {t1:.0f}':>26}")
else:
    print("no RL logs yet")

In [ ]:
# Reasoning dynamics — run 2 charges reasoning, run 3 does not.
# Prediction: run 3 lets reasoning grow (it is free) while final output shrinks.
think_logs = {n: r for n, r in logs.items() if "thinking" in n}

if think_logs:
    for name, recs in think_logs.items():
        q = max(1, len(recs) // 4)
        first, last = recs[:q], recs[-q:]
        r0, r1 = (mean(r["mean_thinking_tokens"] for r in first),
                  mean(r["mean_thinking_tokens"] for r in last))
        t0, t1 = mean(r["mean_tokens"] for r in first), mean(r["mean_tokens"] for r in last)
        print(name)
        print(f"   reasoning {r0:7.1f} -> {r1:7.1f}   ({r1 - r0:+.1f})")
        print(f"   final     {t0 - r0:7.1f} -> {t1 - r1:7.1f}   ({(t1 - r1) - (t0 - r0):+.1f})")
        print(f"   total     {t0:7.1f} -> {t1:7.1f}   ({t1 - t0:+.1f})")
else:
    print("thinking runs not finished yet")

print("\nheld-out reasoning split:")
for name in ("base+think", "sft+think", "rl2+think", "rl3+think"):
    if name in loaded:
        rows = list(loaded[name]["rows"].values())
        reason, total = (mean(r["thinking_tokens"] for r in rows),
                         mean(r["tokens"] for r in rows))
        print(f"  {name:<12} reasoning {reason:6.1f}  final {total - reason:6.1f}"
              f"  total {total:6.1f}")

In [ ]:
# Machine-readable dump + completion check. Paste this output back for analysis.
import subprocess

timings = [json.loads(l) for l in open("checkpoints/timings.jsonl")
           if l.strip()] if Path("checkpoints/timings.jsonl").exists() else []

report = {
    "commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True).stdout.strip(),
    "config": {"rl_steps_nothink": RL_STEPS_NOTHINK, "rl_steps_think": RL_STEPS_THINK,
               "max_new_tokens": MAX_NEW_TOKENS, "max_turns": MAX_TURNS},
    "evals": {name: loaded[name]["summary"] for name in loaded},
    "eval_extra": {
        name: {
            "tokens_on_passing": mean(r["tokens"] for r in d["rows"].values() if r["passed"]),
            "mean_final_tokens": mean(r["tokens"] - r["thinking_tokens"]
                                      for r in d["rows"].values()),
            "hit_token_cap": sum(r["tokens"] >= MAX_NEW_TOKENS for r in d["rows"].values()),
            "passed_task_ids": sorted(t for t, r in d["rows"].items() if r["passed"]),
        } for name, d in loaded.items()},
    "comparisons": comparisons,
    "rl_logs": {name: {"steps": len(recs), "last": recs[-1]} for name, recs in logs.items()},
    "timings_min": {t["stage"]: round(t["seconds"] / 60, 1) for t in timings if t["rc"] == 0},
    "failed_stages": [t["stage"] for t in timings if t["rc"] != 0],
}
Path("checkpoints/results_summary.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

REQUIRED = ["checkpoints/sft/adapter_model.safetensors",
            "checkpoints/eval_base.jsonl", "checkpoints/eval_base_think.jsonl",
            "checkpoints/eval_sft.jsonl", "checkpoints/eval_sft_think.jsonl",
            "checkpoints/rl_run1/final", "checkpoints/rl_run2/final",
            "checkpoints/rl_run3/final", "checkpoints/eval_rl1.jsonl",
            "checkpoints/eval_rl2_think.jsonl", "checkpoints/eval_rl3_think.jsonl"]
missing = [p for p in REQUIRED if not Path(p).exists()]

print("\n" + "=" * 62)
if missing:
    print("INCOMPLETE — still missing:")
    for m in missing:
        print("   ", m)
    print("\nSave this version, then start a new session with its output attached")
    print("(Add Data -> Your Work) and run again to continue.")
else:
    print("PIPELINE COMPLETE — every artifact present.")
print("=" * 62)